# Generate context-specific representations from ChromBERT


**Note**: The remaining examples show command-line usage only (bash).

For the Python API, see [`examples/api/embed_region.ipynb`](../api/embed_region.ipynb).

For Singularity, see [`singularity_use.ipynb`](./singularity_use.ipynb) for `singularity exec` with chrombert-tools.

### generate region embeddings (pretrain and general)

In [4]:
!chrombert-tools embed_region -h

Usage: chrombert-tools embed_region [OPTIONS]

  Generate region embeddings for specified regions or gene promoter regions

Options:
  --region FILE                   Region BED file.
  --gene TEXT                     Gene symbols or IDs separated by ';'.
  --cell-type-bw FILE             Cell type accessibility BigWig file.
  --cell-type-peak FILE           Cell type accessibility Peak BED file.
  --ft-ckpt FILE                  Fine-tuned checkpoint. If provided, skip
                                  fine-tuning.
  --odir DIRECTORY                [default: ./output]
  --oname TEXT                    [default: embedding]
  --genome [hg38|mm10]            [default: hg38]
  --resolution [1kb|200bp|2kb|4kb]
                                  [default: 1kb]
  --mode [fast|full]              Used when training cell-specific model.
                                  [default: fast]
  --batch-size INTEGER            [default: 4]
  --chrombert-cache-dir DIRECTORY
                              

In [18]:
%%bash
chrombert-tools embed_region \
    --region ../data/CTCF_ENCFF664UGR_sample100.bed \
    --odir ./output_emb_region_1kb \
    --genome hg38 \
    --resolution 1kb

Region summary - total: 100, overlapping with ChromBERT: 100 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 0
Using cached region embeddings...

Finished!
Focus region summary - total: 100, overlapping with ChromBERT: 100, non-overlapping: 0
Note: It is possible for a single region to overlap multiple ChromBERT regions.
Overlapping regions BED file: ./output_emb_region_1kb/overlap_region.bed
Non-overlapping regions BED file: ./output_emb_region_1kb/no_overlap_region.bed
Region embeddings saved to: ./output_emb_region_1kb/region_emb_embedding.npy
Embedding type: general


In [10]:
import numpy as np
import pandas as pd

In [12]:
# overlap_region_emb: one 768-dim vector per region

overlap_region_emb = np.load("./output_emb_region_1kb/region_emb_embedding.npy")
print(overlap_region_emb.shape)

# overlap_region.bed: input regions overlapped with ChromBERT's reference regions; contains columns: chrom, start, end, build_region_index

overlap_region = pd.read_csv("./output_emb_region_1kb/overlap_region.bed",sep='\t',header=None, names=['chrom','start','end','build_region_index'])
len(overlap_region),overlap_region.head()

(100, 768)


(100,
    chrom     start       end  build_region_index
 0   chr1  37989946  37990368               32658
 1  chr11   2400199   2400617              289179
 2  chr12   6778809   6779319              391108
 3  chr12  52980788  52981316              424926
 4  chr12  53676021  53676448              425578)

### Generate gene (promoter region) embeddings


In [13]:

%%bash
chrombert-tools embed_region \
    --gene "ENSG00000170921;TANC2;ENSG00000200997;DPYD;SNORA70;tp53;brd4" \
    --odir "./output_emb_genes" \
    --genome hg38 \
    --resolution 1kb



Using cached region embeddings for gene pooling...

Finished!
Note: All gene names were converted to lowercase for matching.
Gene count summary - requested: 7, matched: 7, not found: 0
Matched gene meta saved to: ./output_emb_genes/overlap_genes_meta.tsv
Gene embeddings saved to: ./output_emb_genes/gene_emb_embedding.pkl
Embedding type: general


In [14]:
# gene_emb.pkl: gene embed dict
import pickle
with open("./output_emb_genes/gene_emb_embedding.pkl", "rb") as f:
    gene_emb_dict = pickle.load(f)
for key, value in gene_emb_dict.items():
    print(key, value.shape)


ensg00000170921 (768,)
tanc2 (768,)
ensg00000200997 (768,)
dpyd (768,)
snora70 (768,)
tp53 (768,)
brd4 (768,)


### cell-type-specific embeddings (fine-tune)

In [15]:
# download myoblast 

import subprocess,os
if not os.path.exists('../data/myoblast_ENCFF647RNC_peak.bed'):
    cmd = f'wget https://www.encodeproject.org/files/ENCFF647RNC/@@download/ENCFF647RNC.bed.gz -O ../data/myoblast_ENCFF647RNC_peak.bed.gz'
    subprocess.run(cmd, shell=True)
    cmd = f"gzip -d ../data/myoblast_ENCFF647RNC_peak.bed.gz"
    subprocess.run(cmd, shell=True)

# import subprocess
if not os.path.exists('../data/myoblast_ENCFF149ERN_signal.bigwig'):
    cmd = f'wget https://www.encodeproject.org/files/ENCFF149ERN/@@download/ENCFF149ERN.bigWig -O ../data/myoblast_ENCFF149ERN_signal.bigwig'
    subprocess.run(cmd, shell=True)   

In [16]:
!head -n 100 ../data/myoblast_ENCFF647RNC_peak.bed > ../data/myoblast_ENCFF647RNC_peak_100.bed

In [21]:
%%bash
export CUDA_VISIBLE_DEVICES=0
chrombert-tools embed_region \
    --region ../data/myoblast_ENCFF647RNC_peak_100.bed \
    --odir ./output_cell_specific_emb_train \
    --genome hg38 \
    --resolution 1kb \
    --cell-type-bw ../data/myoblast_ENCFF149ERN_signal.bigwig \
    --cell-type-peak ../data/myoblast_ENCFF647RNC_peak.bed



Preparing dataset ...
Region summary - total: 373422, overlapping with ChromBERT: 368260 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 7920
Total regions: 324690
Fast mode: downsampling to 20k regions
Fine-tuning cell-specific model...

[Attempt 0/2] seed=55
Load pretrained ckpt /mnt/Storage/home/chenqianqian/.cache/chrombert/data/checkpoint/hg38_6k_1kb_pretrain.ckpt successfully!


/mnt/Storage/home/chenqianqian/miniconda3/envs/chrombert/lib/python3.9/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `SpearmanCorrcoef` will save all targets and predictions in the buffer. For large datasets, this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA A100-PCIE-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
/mnt/Storage/home/chenqianqian/miniconda3/envs/chrombert/l

Epoch 0:  20%|██        | 800/4000 [02:18<09:15,  5.76it/s, v_num=0]       
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0:   0%|          | 0/250 [00:00<?, ?it/s]


Metric default_validation/pcc improved. New best score: 0.270


Epoch 0:  40%|████      | 1600/4000 [05:02<07:33,  5.30it/s, v_num=0, default_validation/r2=-0.0472, default_validation/pcc=0.270, default_validation/scc=0.196, default_validation/mae=0.222, default_validation/mse=0.121, default_validation/rmse=0.348, default_validation/mean=0.0806, default_validation/median=0.0757]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 0:  60%|██████    | 2400/4000 [07:46<05:11,  5.14it/s, v_num=0, default_validation/r2=0.0133, default_validation/pcc=0.251, default_validation/scc=0.231, default_validation/mae=0.201, default_validation/mse=0.120, default_validation/rmse=0.346, default_validation/mean=0.118, default_validation/median=0.120]   
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0:   0%|          | 0/250 [00:00<?, ?it/s]


...

Epoch 2:  80%|████████  | 3200/4000 [11:32<02:53,  4.62it/s, v_num=0, default_validation/r2=0.733, default_validation/pcc=0.869, default_validation/scc=0.806, default_validation/mae=0.0985, default_validation/mse=0.0305, default_validation/rmse=0.175, default_validation/mean=0.130, default_validation/median=0.026]
Evaluating the finetuned model performance
Load pretrained ckpt /mnt/Storage/home/chenqianqian/.cache/chrombert/data/checkpoint/hg38_6k_1kb_pretrain.ckpt successfully!
Loading checkpoint from /mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/cli/output_cell_specific_emb_train/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=1-step=126.ckpt
Loading from pl module, remove prefix 'model.'
Loading from pl module, replace 'pretrain_model' with 'pretrain_model.chrombert'
Loaded 111/111 parameters


/mnt/Storage/home/chenqianqian/miniconda3/envs/chrombert/lib/python3.9/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `SpearmanCorrcoef` will save all targets and predictions in the buffer. For large datasets, this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


ft_ckpt: /mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/cli/output_cell_specific_emb_train/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=1-step=126.ckpt, test_metrics: {'pearsonr': 0.8807425498962402, 'spearmanr': 0.8116908073425293, 'mse': 0.03028654120862484, 'mae': 0.10750571638345718, 'r2': 0.7522819668127768}
Attempt metrics: pearsonr=0.8807425498962402
Accepted run (pearsonr=0.8807 >= 0.4).

Finished stage 2: obtained a fine-tuned ChromBERT
Best pearsonr=0.8807425498962402, metrics={'pearsonr': 0.8807425498962402, 'spearmanr': 0.8116908073425293, 'mse': 0.03028654120862484, 'mae': 0.10750571638345718, 'r2': 0.7522819668127768, 'ft_ckpt': '/mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/cli/output_cell_specific_emb_train/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=1-step=126.ckpt'}
Region summary - total: 100, overlapping with

100%|██████████| 26/26 [00:02<00:00, 12.38it/s]



Finished!
Focus region summary - total: 100, overlapping with ChromBERT: 101, non-overlapping: 0
Note: It is possible for a single region to overlap multiple ChromBERT regions.
Overlapping regions BED file: ./output_cell_specific_emb_train/overlap_region.bed
Non-overlapping regions BED file: ./output_cell_specific_emb_train/no_overlap_region.bed
Region embeddings saved to: ./output_cell_specific_emb_train/region_emb_embedding.npy
Embedding type: cell-specific


In [24]:
# region_emb.npy: one 768-dim vector per region
import numpy as np
import pandas as pd
overlap_region_emb = np.load("./output_cell_specific_emb_train/region_emb_embedding.npy")
print(overlap_region_emb.shape)

# overlap_region.bed: input regions overlapped with ChromBERT's reference regions; contains columns: chrom, start, end, build_region_index

overlap_region = pd.read_csv("./output_cell_specific_emb_train/overlap_region.bed",sep='\t',header=None, names=['chrom','start','end','build_region_index'])
len(overlap_region),overlap_region.head()

(101, 768)


(101,
   chrom   start     end  build_region_index
 0  chr1  180791  180871                  38
 1  chr1  181400  181580                  39
 2  chr1  182681  182820                  40
 3  chr1  191400  191540                  46
 4  chr1  268011  268080                  54)

### cell-type-specific embeddings (loading fine-tuned model)

In [25]:
import glob
ft_ckpt_dir = "./output_cell_specific_emb_train/train/**/*.ckpt"

ft_ckpt = glob.glob(ft_ckpt_dir, recursive=True)[0]
ft_ckpt

'./output_cell_specific_emb_train/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=1-step=126.ckpt'

In [26]:


!chrombert-tools embed_region \
--region ../data/myoblast_ENCFF647RNC_peak_100.bed \
--odir ./output_cell_specific_emb_load_ckpt \
--genome hg38 \
--resolution 1kb \
--ft-ckpt {ft_ckpt}

Using provided fine-tuned checkpoint: ./output_cell_specific_emb_train/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=1-step=126.ckpt
Load pretrained ckpt /mnt/Storage/home/chenqianqian/.cache/chrombert/data/checkpoint/hg38_6k_1kb_pretrain.ckpt successfully!
Loading checkpoint from ./output_cell_specific_emb_train/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=1-step=126.ckpt
Loading from pl module, remove prefix 'model.'
Loading from pl module, replace 'pretrain_model' with 'pretrain_model.chrombert'
Loaded 111/111 parameters
Region summary - total: 100, overlapping with ChromBERT: 101 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 0
Your supervised_file does not contain the 'label' column. Please verify whether ground truth column ('label') is required. If it is not needed, you may disregard this message.
100%|████

In [28]:
# region_emb.npy: one 768-dim vector per region
overlap_region_emb2 = np.load("./output_cell_specific_emb_load_ckpt/region_emb_embedding.npy")
print(overlap_region_emb.shape)

# overlap_region.bed: input regions overlapped with ChromBERT's reference regions; contains columns: chrom, start, end, build_region_index

overlap_region2 = pd.read_csv("./output_cell_specific_emb_load_ckpt/overlap_region.bed",sep='\t',header=None, names=['chrom','start','end','build_region_index'])
len(overlap_region),overlap_region.head()

(101, 768)


(101,
   chrom   start     end  build_region_index
 0  chr1  180791  180871                  38
 1  chr1  181400  181580                  39
 2  chr1  182681  182820                  40
 3  chr1  191400  191540                  46
 4  chr1  268011  268080                  54)

In [30]:
assert (overlap_region_emb2 == overlap_region_emb).all()